In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import zscore

In [2]:
# Load the dataset from CSV file and define column names
df = pd.read_csv('Biometrics_4HZ/labeled_combined_biometric_data_4HZ.csv', low_memory=False)
df.columns = ['acc_x','acc_y','acc_z','bvp','eda', 'hr', 'SPO2','Pulse_Rate', 'motion','temp','Subject','Session','Label']

In [3]:
# Convert columns to numeric if possible
df[['SPO2', 'Pulse_Rate']] = df[['SPO2', 'Pulse_Rate']].apply(pd.to_numeric, errors='coerce')

In [4]:
# Interpolate NaN values for SPO2 and Pulse_rate subject-session wise
df[['SPO2', 'Pulse_Rate']] = df.groupby(['Subject', 'Session'])[['SPO2', 'Pulse_Rate']].apply(
    lambda group: group.interpolate(method='linear', limit_direction='forward')
).reset_index(drop=True)[['SPO2', 'Pulse_Rate']]

### Boxplot of measurements by Subject and Drowsiness Level ###

In [ ]:
def map_drowsiness_level(x):
    if x < 4:
        return 'Low'
    elif 4 <= x <= 6:
        return 'Med'
    else:
        return 'High'

# Apply the drowsiness level mapping
df['DrowsinessLevel'] = df['Label'].apply(map_drowsiness_level)

# List of measurement columns to plot
measurement_columns = ['eda', 'temp', 'hr', 'acc_x', 'acc_y', 'acc_z', 'bvp', 'SPO2', 'Pulse_Rate', 'motion']

# Remove outliers for each user in each measurement column using z-score
for measurement in measurement_columns:
    # For each user and signal, calculate the Z-score
    df[measurement] = df.groupby('Subject')[measurement].transform(lambda x: zscore(x, nan_policy='omit'))
    
    # Remove rows where Z-score exceeds 3 (threshold for outliers)
    df = df[np.abs(df[measurement]) < 3]

# Standardize the data (Z-score standardization) for each signal
for measurement in measurement_columns:
    df[measurement] = df.groupby('Subject')[measurement].transform(lambda x: (x - x.mean()) / x.std())

# Set the aesthetic style of the plots
sns.set(style="whitegrid")

# Loop over each measurement column
for measurement in measurement_columns:
    plt.figure(figsize=(15, 8))
    
    # Create a boxplot for each subject and drowsiness level
    sns.boxplot(
        x='Subject',
        y=measurement,
        hue='DrowsinessLevel',
        data=df,
        order=sorted(df['Subject'].unique()),
        hue_order=['Low', 'Med', 'High'],
        showfliers=False
    )
    
    # Customize the plot
    plt.title(f'Boxplot of {measurement} by Subject and Drowsiness Level', fontsize=24)
    plt.xlabel('Subject', fontsize=18)
    plt.ylabel(measurement, fontsize=18)
    
  
    # For other measurements, place the legend in the upper left
    plt.legend(
        title='Drowsiness Level',
        loc='upper left', 
        fontsize=15,  
        title_fontsize=16, 
            frameon=False  
        )
    
    plt.xticks(rotation=90, fontsize=12)
    plt.tight_layout()
    plt.show()
